# AI4Mars majority-class baseline

This optional notebook calculates the majority-class comparison independently. It does not train the U-Net. Run the setup, data, and baseline sections in order.


In [ ]:
%pip install -q datasets scikit-learn pillow matplotlib

import numpy as np
import torch
from pathlib import Path

SEED = 0
DATASET_ID = "hassanjbara/AI4MARS"
IMAGE_SIZE = (192, 192)
NUM_CLASSES = 4
IGNORE_INDEX = 255
CLASS_NAMES = ["Soil", "Bedrock", "Sand", "Big rock"]
LABEL_ID_MAPPING = {0: 0, 1: 1, 2: 2, 3: 3}
BATCH_SIZE = 16
NUM_WORKERS = 2

np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Runtime device: {device}")


## 1. Load and split the dataset

This scans masks to exclude unusable samples and creates the same reproducible 80/20 split used by the training notebook.


In [ ]:
# Dataset loading, cleaning, and splitting

from datasets import load_dataset, Image as HFImage
from sklearn.model_selection import train_test_split
from io import BytesIO
from PIL import Image


# Load the dataset.
train_data = load_dataset(DATASET_ID, split="train")

# Inspect encoded image fields without decoding every image automatically.
raw_data = train_data.select_columns(
    ["image", "label_mask", "has_labels"]
).cast_column(
    "image", HFImage(decode=False)
).cast_column(
    "label_mask", HFImage(decode=False)
)

def decode_image_field(value):
    if value is None:
        return None

    if value.get("bytes") is not None:
        return Image.open(BytesIO(value["bytes"])).copy()

    if value.get("path"):
        return Image.open(value["path"]).copy()

    raise ValueError("Image has neither bytes nor a path.")

usable_indices = []
bad_image_indices = []
missing_mask_indices = []
empty_mask_indices = []

for idx in range(len(raw_data)):
    row = raw_data[idx]

    try:
        image = decode_image_field(row["image"])
        image.close()
    except Exception:
        bad_image_indices.append(idx)
        continue

    if not row["has_labels"] or row["label_mask"] is None:
        missing_mask_indices.append(idx)
        continue

    mask = decode_image_field(row["label_mask"]).convert("L")
    valid_pixels = np.isin(np.asarray(mask), list(LABEL_ID_MAPPING)).any()
    mask.close()

    if not valid_pixels:
        empty_mask_indices.append(idx)
        continue

    usable_indices.append(idx)

train_indices, val_indices = train_test_split(
    usable_indices,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

train_subset = train_data.select(train_indices)
val_subset = train_data.select(val_indices)

print("Total records:", len(train_data))
print("Usable records:", len(usable_indices))
print("Broken images:", len(bad_image_indices))
print("Missing masks:", len(missing_mask_indices))
print("Empty masks:", len(empty_mask_indices))
print("Training:", len(train_subset))
print("Validation:", len(val_subset))

assert set(train_indices).isdisjoint(val_indices)
assert len(train_indices) + len(val_indices) == len(usable_indices)

## 2. Build data loaders


In [ ]:
# PyTorch dataset and data loaders

import torch
from torch.utils.data import Dataset, DataLoader

class MarsTerrainDataset(Dataset):
    def __init__(self, hf_dataset, image_size=IMAGE_SIZE):
        self.data = hf_dataset
        self.image_size = image_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]

        image = row["image"].convert("L")
        mask = row["label_mask"].convert("L")

        image = image.resize(
            self.image_size,
            Image.Resampling.BILINEAR,
        )
        raw_mask = np.asarray(mask, dtype=np.uint8)
        mapped_mask = np.full(raw_mask.shape, IGNORE_INDEX, dtype=np.uint8)
        for source_id, class_id in LABEL_ID_MAPPING.items():
            mapped_mask[raw_mask == source_id] = class_id
        mask = Image.fromarray(mapped_mask, mode="L").resize(
            self.image_size, Image.Resampling.NEAREST
        )

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        mask_array = np.asarray(mask, dtype=np.int64)

        x = torch.from_numpy(image_array.copy()).unsqueeze(0)
        y = torch.from_numpy(mask_array.copy())

        return x, y


train_dataset = MarsTerrainDataset(train_subset)
val_dataset = MarsTerrainDataset(val_subset)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

# Sanity check one batch.
images, masks = next(iter(train_loader))

print("Image batch:", images.shape)
print("Mask batch:", masks.shape)
print("Mask values:", torch.unique(masks).tolist())

assert images.shape[1:] == (1, *IMAGE_SIZE)
assert masks.shape[1:] == IMAGE_SIZE
assert torch.all(
    (masks == IGNORE_INDEX) |
    ((masks >= 0) & (masks < NUM_CLASSES))
)

print("Dataset pipeline checks passed.")

## 3. Calculate the majority-class baseline

The most common class is selected from training pixels only, then evaluated on validation pixels.


In [ ]:
# Majority-class baseline: always predict the most common training label

import torch

# 1. Find the majority class from TRAINING masks only
training_class_pixel_counts = torch.zeros(NUM_CLASSES, dtype=torch.long)

for _, masks in train_loader:
    for class_id in range(NUM_CLASSES):
        training_class_pixel_counts[class_id] += (masks == class_id).sum().item()

majority_class_id = training_class_pixel_counts.argmax().item()

print("Training pixel distribution:")
for class_id, name in enumerate(CLASS_NAMES):
    percentage = (
        training_class_pixel_counts[class_id].item()
        / training_class_pixel_counts.sum().item()
        * 100
    )
    print(f"{name:10s}: {percentage:6.2f}%")

print(f"\nMajority class: {CLASS_NAMES[majority_class_id]}")


# 2. Evaluate the baseline on validation data
baseline_confusion_matrix = torch.zeros(
    NUM_CLASSES, NUM_CLASSES, dtype=torch.long
)

with torch.no_grad():
    for _, masks in val_loader:
        targets = masks.cpu().long()

        # Exclude unlabeled pixels
        valid = targets != IGNORE_INDEX
        targets = targets[valid]

        if targets.numel() == 0:
            continue

        # Predict the majority class for every valid pixel
        predictions = torch.full_like(targets, majority_class_id)

        # Rows = true classes, columns = predicted classes
        indices = targets * NUM_CLASSES + predictions
        baseline_confusion_matrix += torch.bincount(
            indices,
            minlength=NUM_CLASSES**2
        ).reshape(NUM_CLASSES, NUM_CLASSES)


# 3. Calculate metrics
true_positive_pixels = baseline_confusion_matrix.diag().float()
actual_class_pixels = baseline_confusion_matrix.sum(dim=1).float()
predicted_class_pixels = baseline_confusion_matrix.sum(dim=0).float()

class_union = actual_class_pixels + predicted_class_pixels - true_positive_pixels

baseline_per_class_iou = torch.full((NUM_CLASSES,), float("nan"))
present_classes = class_union > 0
baseline_per_class_iou[present_classes] = (
    true_positive_pixels[present_classes] / class_union[present_classes]
)

total_pixels = baseline_confusion_matrix.sum().item()
baseline_pixel_accuracy = true_positive_pixels.sum().item() / total_pixels
baseline_mean_iou = baseline_per_class_iou[present_classes].mean().item()


# 4. Display results
print("\n=== Majority-Class Baseline Results ===")
print(f"Pixel Accuracy: {baseline_pixel_accuracy:.4f}")
print(f"Mean IoU:       {baseline_mean_iou:.4f}")

print("\nPer-class IoU:")
for class_id, name in enumerate(CLASS_NAMES):
    value = baseline_per_class_iou[class_id].item()
    if torch.isnan(baseline_per_class_iou[class_id]):
        print(f"{name:10s}: N/A")
    else:
        print(f"{name:10s}: {value:.4f}")

print("\nConfusion Matrix (rows=true, columns=predicted):")
print(baseline_confusion_matrix)
